# 03 · Treinamento dos Modelos
Compara uma abordagem **supervisionada progressiva**: baseline linear → ensemble → gradient boosting com ajuste de hiperparâmetros. Os dados já vêm tratados de `02_preprocessing.ipynb`.

In [1]:
import pandas as pd
import numpy as np
import pickle, json
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

In [2]:
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()
print(X_train.shape, X_test.shape)

(198608, 30) (85118, 30)


## Modelo 1 — Regressão Logística (baseline)

O `StandardScaler` fica dentro do `Pipeline`, então ele só aprende (fit) com `X_train` — corrige o vazamento de dados da primeira versão do notebook, em que o scaler era ajustado no dataframe inteiro antes do split.

In [3]:
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])
lr_pipeline.fit(X_train, y_train)
probs_lr = lr_pipeline.predict_proba(X_test)[:, 1]
pred_lr = lr_pipeline.predict(X_test)
print(classification_report(y_test, pred_lr))
print('ROC-AUC:', roc_auc_score(y_test, probs_lr))
print('PR-AUC :', average_precision_score(y_test, probs_lr))

              precision    recall  f1-score   support

           0       1.00      0.97      0.99     84976
           1       0.05      0.89      0.10       142

    accuracy                           0.97     85118
   macro avg       0.53      0.93      0.54     85118
weighted avg       1.00      0.97      0.98     85118

ROC-AUC: 0.9629068837332033
PR-AUC : 0.6910931531626647


**Leitura**: com `class_weight='balanced'`, o baseline linear consegue recall alto, mas ao custo de muitos falsos positivos (precision baixa) — típico de modelo linear numa fronteira de decisão não-linear. Serve como piso de comparação honesto.

## Modelo 2 — Random Forest

In [4]:
rf = RandomForestClassifier(n_estimators=100, max_depth=12, class_weight='balanced',
                             n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
probs_rf = rf.predict_proba(X_test)[:, 1]
pred_rf = rf.predict(X_test)
print(classification_report(y_test, pred_rf))
print('ROC-AUC:', roc_auc_score(y_test, probs_rf))
print('PR-AUC :', average_precision_score(y_test, probs_rf))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     84976
           1       0.88      0.75      0.81       142

    accuracy                           1.00     85118
   macro avg       0.94      0.87      0.90     85118
weighted avg       1.00      1.00      1.00     85118

ROC-AUC: 0.9453489021589525
PR-AUC : 0.8075170809230464


## Modelo 3 — XGBoost (baseline)

In [5]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb_base = XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss',
                          random_state=42, n_jobs=-1)
xgb_base.fit(X_train, y_train)
probs_xgb = xgb_base.predict_proba(X_test)[:, 1]
pred_xgb = xgb_base.predict(X_test)
print(classification_report(y_test, pred_xgb))
print('ROC-AUC:', roc_auc_score(y_test, probs_xgb))
print('PR-AUC :', average_precision_score(y_test, probs_xgb))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     84976
           1       0.91      0.76      0.83       142

    accuracy                           1.00     85118
   macro avg       0.95      0.88      0.91     85118
weighted avg       1.00      1.00      1.00     85118

ROC-AUC: 0.9759677380324122
PR-AUC : 0.8097490641386494


## Ajuste de hiperparâmetros (GridSearchCV)

**Correção importante**: na primeira versão, a grade de parâmetros tinha uma chave escrita como `"max=depth"` em vez de `"max_depth"`. O scikit-learn/XGBoost não reclama de chave desconhecida em `**kwargs` — ele simplesmente ignora o parâmetro, com um aviso (`warning`) que passa despercebido. Resultado: o grid search **nunca variou `max_depth` de verdade**, e o "melhor modelo" reportado não refletia nenhuma otimização real desse hiperparâmetro. Corrigido abaixo, junto com a métrica de scoring trocada de `recall` (que sozinha incentivaria threshold degenerado) para `average_precision` (PR-AUC), mais adequada para classes raras.

In [6]:
param_grid = {
    'max_depth': [3, 5],
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
}

grid = GridSearchCV(
    XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss',
                  random_state=42, n_jobs=-1),
    param_grid, scoring='average_precision', cv=3, n_jobs=-1
)
grid.fit(X_train, y_train)
print('Melhores hiperparâmetros:', grid.best_params_)
best_xgb = grid.best_estimator_

Melhores hiperparâmetros: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}


In [7]:
probs_best = best_xgb.predict_proba(X_test)[:, 1]
pred_best = best_xgb.predict(X_test)
print(classification_report(y_test, pred_best))
print('ROC-AUC:', roc_auc_score(y_test, probs_best))
print('PR-AUC :', average_precision_score(y_test, probs_best))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     84976
           1       0.87      0.77      0.82       142

    accuracy                           1.00     85118
   macro avg       0.94      0.88      0.91     85118
weighted avg       1.00      1.00      1.00     85118

ROC-AUC: 0.9655693587717229
PR-AUC : 0.8140806480535712


## Persistência dos modelos treinados

Todos os modelos são salvos para reprodutibilidade e para a etapa de avaliação comparativa em `04_avaliacao.ipynb` — nenhum modelo é descartado silenciosamente (diferente da primeira versão, em que undersampling/SMOTE eram calculados e nunca chegavam a alimentar um modelo).

In [8]:
import os
os.makedirs('../models', exist_ok=True)

with open('../models/logistic_regression.pkl', 'wb') as f:
    pickle.dump(lr_pipeline, f)
with open('../models/random_forest.pkl', 'wb') as f:
    pickle.dump(rf, f)
with open('../models/xgboost_baseline.pkl', 'wb') as f:
    pickle.dump(xgb_base, f)
with open('../models/xgboost_tuned.pkl', 'wb') as f:
    pickle.dump(best_xgb, f)

print('Modelos salvos em models/')

Modelos salvos em models/


Segue para `04_avaliacao.ipynb`.